# Modeling

Packages and setup

In [ ]:
# Imports & settings
from imports import *
notebook_settings() 
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6 = DATA_DIR_3_x

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data_all = load_ai_labeled_sales()
customers_before_after = pd.read_pickle(DATA_DIR_3 / 'customers_before_after.pkl')

In [ ]:
takeout_list = [
    ['to','go'],
    ['take','out'],
    ['pick','up'],
    ['take','away'],
    ['carry','out'],
    ['mobile','order'],
    ['t/o'],
    ['delivery'],
    ['grab and go'],
    ['order ahead'],
    ['meal prep']]
takeout = '|'.join(sep.join(s) for sep in ['',' ','-'] for s in takeout_list)

total = 0
for loc_id in location_ids_by_coverage:
    total += data_all[loc_id].shape[0]
print(total)

total = 0
for loc_id in location_ids_by_coverage:
    total += (data_all[loc_id]
              .fillna({'item_modifications':''})
              .query('~item_modifications.str.lower().str.contains(@takeout)')
              .query('~item_name.str.lower().str.contains(@takeout)')
              .shape[0])
print(total)

data = {}
for loc_id in location_ids_by_coverage:
    df = (
        data_all[loc_id]
        .query('~is_drink')
        .query('~is_nonfood')
        .query('~is_nonmeal_merchandise')
        )

    
    df.to_parquet(f'{DATA_DIR_3_5}/{loc_id}.parquet',index=False)
    
    df = (
        df
        .fillna({'item_modifications':''})
        .query('~item_modifications.str.lower().str.contains(@takeout)')
        .query('~item_name.str.lower().str.contains(@takeout)'))
    
    data[loc_id] = df
    
    df.to_parquet(f'{DATA_DIR_3_6}/{loc_id}.parquet',index=False)

Putting in customer data

In [ ]:
# total_transactions = 0
# total_menu_items = 0
# total_transactions_1 = 0
# total_menu_items_1 = 0
# first_day = pd.to_datetime('2024-01-01')
# last_day = pd.to_datetime('2000-01-01')
# for loc_id in location_ids_by_coverage:
#     df1 = data[loc_id]
    
#     if loc_id == 'VLZX7K2M9QD4T':
#         total_transactions += df1['transaction_id'].nunique()
#     else:
#         total_transactions += df1['order_id'].nunique()
#     total_menu_items += df1['item_name'].nunique()
    
#     if df1.index[0].tz_localize(None) < first_day:
#         first_day = df1.index[0].tz_localize(None)
#     if df1.index[-1].tz_localize(None) > last_day:
#         last_day = df1.index[-1].tz_localize(None)
    
#     df2 = sales_menu_customers_data[loc_id]
    
#     if loc_id == 'VLZX7K2M9QD4T':
#         total_transactions_1 += df2['transaction_id'].nunique()
#     else:
#         total_transactions_1 += df2['order_id'].nunique()
#     total_menu_items_1 += df2['item_name'].nunique()

# print(f"Total transactions (sales_and_menu_data): {total_transactions:,}")
# print(f"Total menu items (sales_and_menu_data): {total_menu_items:,}")
# print(f"Total transactions (sales_menu_customers_data): {total_transactions_1:,}")
# print(f"Total menu items (sales_menu_customers_data): {total_menu_items_1:,}")
# print(f"First day: {first_day.strftime('%Y-%m-%d')}")
# print(f"Last day: {last_day.strftime('%Y-%m-%d')}")

Roll data

In [ ]:
def season_from_month(month):
    return 'winter' if month in [12, 1, 2] else \
           'spring' if month in [3, 4, 5] else \
           'summer' if month in [6, 7, 8] else 'fall'

def weighted_avg(window):
    return (window['item_quantity'] * window['unit_price']).sum() / window['item_quantity'].sum()

def rolling_window_avg(df_, filter_col, column, name, lookback_period, lookback_unit):
    df = (df_
          #.query(filter)
          .assign(weighted_val = lambda df: df[filter_col] * df[column])
          ['weighted_val']
          .rolling(f'{lookback_period}{lookback_unit}')
          .sum()
          .rename(name)
          .to_frame()
          .reset_index()
          .drop_duplicates('created_at')
          .set_index('created_at')
          .shift(1)
          .ffill()
          .bfill()
          [name]
          )
    return df

In [ ]:
lookback_period = 1
lookback_unit = 'D'

hour_mapping = {
    22: -1, 23: -1, 
    1: -1, 6: -1, 7: -1
}

model_df_list = []
for loc_id in location_ids_by_coverage:
    
    df = data[loc_id].set_index('created_at').assign(meat = lambda df: ~df.vegetarian)
    
    model_df = (df
                .assign(
                    hour_of_day = lambda df: df.index.to_series().dt.hour.replace(hour_mapping).astype("category"),
                    day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
                    weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
                    meal_period = lambda df: pd.cut(df.index.to_series().dt.hour.astype("category"), 
                                        bins=[0, 5, 11, 16, 22, 24], 
                                        labels=['Late', 'Breakfast', 'Lunch', 'Dinner', 'Late'], 
                                        right=False,
                                        ordered=False).astype(str).replace({'Late': 'Dinner'}).astype("category"),
                    day_of_month = lambda df: df.index.to_series().dt.day.astype("category"),
                    month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
                    month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
                    season = lambda df: df.index.month.map(season_from_month).astype("category"),
                    year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
                    year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
                    date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes)
                .reset_index()
                .rename(columns={'created_at':'created_at_tz'})
                .assign(created_at_tz = lambda df: df['created_at_tz'].dt.tz_convert('UTC'))
                .rename(columns={'created_at_tz':'created_at'})
                .set_index('created_at')
                .join([rolling_window_avg(df, 'meat', 'item_price', 'meat_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'meat', 'item_quantity', 'meat_window_quantity', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegetarian', 'item_price', 'vegetarian_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegetarian', 'item_quantity', 'vegetarian_window_quantity', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegan', 'item_price', 'vegan_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegan', 'item_quantity', 'vegan_window_quantity', lookback_period, lookback_unit)],
                      how='left')
                .assign(
                    vegan_window_avg = lambda df: (df['vegan_window_price'] / df['vegan_window_quantity']).ffill().bfill(),
                    vegetarian_window_avg = lambda df: (df['vegetarian_window_price'] / df['vegetarian_window_quantity']).ffill().bfill(),
                    meat_window_avg = lambda df: (df['meat_window_price'] / df['meat_window_quantity']).ffill().bfill(),
                    vegan_outcome = lambda df: 1*df['vegan'],
                    vegetarian_outcome = lambda df: 1*df['vegetarian'],
                    total_outcome = 1,
                    nonvegan_outcome = lambda df: 1 - df['vegan_outcome'],
                    meat_outcome = lambda df: 1 - df['vegetarian_outcome'],
                    chicken_fish_outcome = lambda df: 1*df['chicken_fish'])
                #.pipe(lambda df: print(df['item_quantity'].sum()) or df)
                #.pipe(lambda df: print(loc_id) or df)
                #.pipe(lambda df: print(df.shape) or df)
                .reset_index()
                .loc[lambda df: df.index.repeat(df['item_quantity'])]
                .set_index('created_at')
                .assign(item_quantity = 1)
                #.pipe(lambda df: print(df['item_quantity'].sum()) or df)
                #.pipe(lambda df: print(df.shape) or df)
                )
    #if loc_id in ['SRQS8F7JWA9MZ', '2HRX9P6HKXA8V']:
    model_df_list.append(model_df)
    
model_data = pd.concat(model_df_list).assign(location_id = lambda df: df['location_id'].astype('category'))

Aggregate data

In [ ]:
model_data_customers = pd.merge(
    model_data.drop(columns=['_merge']).reset_index(),
    customers_before_after.explode('customer_ids'),
    left_on=['location_id','customer_id'], right_on=['loc_id','customer_ids'], how='inner', indicator=True).set_index('created_at')

In [ ]:
def clip_na(group):
    valid_part = group.dropna(subset='vegan_window_avg')
    first_valid = valid_part.index.min()
    last_valid = valid_part.index.max()
    clipped = group.loc[first_valid:last_valid]
    return clipped

# # Aggregate
# daily_model_data = (model_data
#                     .groupby(['location_id', model_data.index.normalize()], observed=True)
#                     .agg({# Price averages
#                           'item_quantity':'sum',
#                           'vegan_window_quantity':'sum',
#                           'vegetarian_window_quantity':'sum',
#                           'meat_window_quantity':'sum',
#                           'vegan_window_avg':'mean',
#                           'vegetarian_window_avg':'mean',
#                           'meat_window_avg':'mean',
                          
#                           # Outcomes
#                           'vegan_outcome':'sum',
#                           'vegetarian_outcome':'sum',
#                           'total_outcome':'sum',
#                           'nonvegan_outcome':'sum',
#                           'meat_outcome':'sum',
#                           'chicken_fish_outcome':'sum'})
#                     .rename(columns={'item_quantity':'item_quantity_day'})
#                     .rename_axis(index=['location_id','created_at'])
#                     .reindex(pd.MultiIndex.from_product([location_ids_by_coverage,
#                                                          pd.date_range(model_data.index.min().normalize(),
#                                                                        model_data.index.max().normalize(),
#                                                                        freq='D', tz='UTC')], 
#                                                         names=['location_id', 'created_at']))
#                     .reset_index()
#                     .set_index('created_at')
#                     .groupby('location_id', group_keys=False) # Switch argument in future version of Pandas
#                     .apply(clip_na)
#                     .fillna({'chicken_fish_outcome':0,
#                              'nonvegan_outcome':0, 
#                              'meat_outcome':0,
#                              'vegan_outcome':0, 
#                              'vegetarian_outcome':0,
#                              'total_outcome':0})
#                     #.query('0 < item_quantity_day')
#                     .assign(
#                         vegan_window_avg = lambda df: df.groupby('location_id')['vegan_window_avg'].ffill(),
#                         vegetarian_window_avg = lambda df: df.groupby('location_id')['vegetarian_window_avg'].ffill(),
#                         meat_window_avg = lambda df: df.groupby('location_id')['meat_window_avg'].ffill(),                        
#                         day_of_week_cat = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
#                         day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category").cat.codes,
#                         weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
#                         day_of_month_cat = lambda df: df.index.to_series().dt.day.astype("category"),
#                         day_of_month = lambda df: df.index.to_series().dt.day.astype("category").cat.codes,
#                         month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
#                         month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
#                         season = lambda df: df.index.month.map(season_from_month).astype("category"),
#                         year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
#                         year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
#                         date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes,
#                         exposure_VLZX7K2M9QD4T_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'VLZX7K2M9QD4T') & (pd.to_datetime(before_after_details_true.loc['VLZX7K2M9QD4T', 'cross_over_date']).tz_convert('UTC') <= df.index),
#                             1),
#                         exposure_SRQS8F7JWA9MZ_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime('2020-06-25').tz_localize('UTC') <= df.index),
#                             1),
#                         exposure_SRQS8F7JWA9MZ_2 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime(before_after_details_true.loc['SRQS8F7JWA9MZ', 'cross_over_date']).tz_convert('UTC') <= df.index), # 2020-09-12 13:48:10-04:00
#                             1),
#                         exposure_2HRX9P6HKXA8V_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == '2HRX9P6HKXA8V') & (pd.to_datetime(before_after_details_true.loc['2HRX9P6HKXA8V', 'cross_over_date']).tz_convert('UTC') <= df.index),
#                             1),
#                         exposure_JHDN7CF1C03X5_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime(before_after_details_true.loc['JHDN7CF1C03X5', 'cross_over_date']).tz_convert('UTC') <= df.index),
#                             1),
#                         exposure_JHDN7CF1C03X5_2 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime('2020-03-12').tz_localize('UTC') <= df.index),
#                             1),
#                         exposure_L69HYJ4Y3TR91_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'L69HYJ4Y3TR91') & (pd.to_datetime(before_after_details_true.loc['L69HYJ4Y3TR91', 'cross_over_date']).tz_convert('UTC') <= df.index),
#                             1),
#                         exposure_ED5J990H5VAZT_1 = lambda df: pd.Series(0, index=df.index).mask(
#                             (df['location_id'] == 'ED5J990H5VAZT') & (pd.to_datetime(before_after_details_true.loc['ED5J990H5VAZT', 'cross_over_date']).tz_convert('UTC') <= df.index),
#                             1)
#                         )
#                     )

# Aggregate
daily_model_data_customers = (model_data_customers
                    .groupby(['location_id', model_data_customers.index.normalize()], observed=True)
                    .agg({# Price averages
                          'item_quantity':'sum',
                          'vegan_window_quantity':'sum',
                          'vegetarian_window_quantity':'sum',
                          'meat_window_quantity':'sum',
                          'vegan_window_avg':'mean',
                          'vegetarian_window_avg':'mean',
                          'meat_window_avg':'mean',
                          
                          # Outcomes
                          'vegan_outcome':'sum',
                          'vegetarian_outcome':'sum',
                          'total_outcome':'sum',
                          'nonvegan_outcome':'sum',
                          'meat_outcome':'sum',
                          'chicken_fish_outcome':'sum'})
                    .rename(columns={'item_quantity':'item_quantity_day'})
                    .rename_axis(index=['location_id','created_at'])
                    .reindex(pd.MultiIndex.from_product([location_ids_by_coverage,
                                                         pd.date_range(model_data_customers.index.min().normalize(),
                                                                       model_data_customers.index.max().normalize(),
                                                                       freq='D', tz='UTC')], 
                                                        names=['location_id', 'created_at']))
                    .reset_index()
                    .set_index('created_at')
                    .groupby('location_id', group_keys=False) # Switch argument in future version of Pandas
                    .apply(clip_na)
                    .fillna({'chicken_fish_outcome':0,
                             'nonvegan_outcome':0, 
                             'meat_outcome':0,
                             'vegan_outcome':0, 
                             'vegetarian_outcome':0,
                             'total_outcome':0})
                    #.query('0 < item_quantity_day')
                    .assign(
                        vegan_window_avg = lambda df: df.groupby('location_id')['vegan_window_avg'].ffill(),
                        vegetarian_window_avg = lambda df: df.groupby('location_id')['vegetarian_window_avg'].ffill(),
                        meat_window_avg = lambda df: df.groupby('location_id')['meat_window_avg'].ffill(),                        
                        day_of_week_cat = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
                        day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category").cat.codes,
                        weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
                        day_of_month_cat = lambda df: df.index.to_series().dt.day.astype("category"),
                        day_of_month = lambda df: df.index.to_series().dt.day.astype("category").cat.codes,
                        month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
                        month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
                        season = lambda df: df.index.month.map(season_from_month).astype("category"),
                        year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
                        year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
                        date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes,
                        exposure_VLZX7K2M9QD4T_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'VLZX7K2M9QD4T') & (pd.to_datetime(before_after_details_true.loc['VLZX7K2M9QD4T', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_SRQS8F7JWA9MZ_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime('2020-06-25').tz_localize('UTC') <= df.index),
                            1),
                        exposure_SRQS8F7JWA9MZ_2 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime(before_after_details_true.loc['SRQS8F7JWA9MZ', 'cross_over_date']).tz_convert('UTC') <= df.index), # 2020-09-12 13:48:10-04:00
                            1),
                        exposure_2HRX9P6HKXA8V_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == '2HRX9P6HKXA8V') & (pd.to_datetime(before_after_details_true.loc['2HRX9P6HKXA8V', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_JHDN7CF1C03X5_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime(before_after_details_true.loc['JHDN7CF1C03X5', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_JHDN7CF1C03X5_2 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime('2020-03-12').tz_localize('UTC') <= df.index),
                            1),
                        exposure_L69HYJ4Y3TR91_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'L69HYJ4Y3TR91') & (pd.to_datetime(before_after_details_true.loc['L69HYJ4Y3TR91', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_ED5J990H5VAZT_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'ED5J990H5VAZT') & (pd.to_datetime(before_after_details_true.loc['ED5J990H5VAZT', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1)
                        )
                    )

# Put in location data: locations have data for cuisine, etc.
#daily_model_data = pd.merge(daily_model_data, locations, left_on='location_id', right_index=True, how='left')

# Export
# daily_model_data.to_parquet("data/4_data_parquet_modeling/all_locations_daily.parquet")

# Put in location data: locations have data for cuisine, etc.
daily_model_data_customers = pd.merge(daily_model_data_customers, locations, left_on='location_id', right_index=True, how='left')

# Export
daily_model_data_customers.to_parquet("data/4_data_parquet_modeling/all_locations_daily_customers.parquet")

Plotting tool

In [ ]:
# Plot function for resampling and visualization
def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
    resampled_pred = data.resample(freq)['pred'].mean()
    resampled_actual = data.resample(freq)['vegan_outcome'].mean()

    if start_date and end_date:
        resampled_pred = resampled_pred.loc[start_date:end_date]
        resampled_actual = resampled_actual.loc[start_date:end_date]

    resampled_pred.plot(color='orange', label='Predicted', title=title)
    resampled_actual.plot(color='blue', alpha=0.3, label='Actual', title=title)

In [ ]:
before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                 .set_index('loc_id')
                                 ['customer_ids']
                                 .loc[location_ids_by_coverage[1:]])

ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
axes = axes.flatten()
for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off') # Turn off the axis if no data.
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off') # Turn off the axis if no data.
        continue

    customer_mean_day = customer_orders_day.groupby('customer_id').transform('mean').round().astype(int)
    deviations = customer_orders_day.sub(customer_mean_day)
    deviation_counts = deviations.value_counts().sort_index()

    ax.bar(deviation_counts.index, deviation_counts.values, color='skyblue', edgecolor='black')
    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-20.5, 20.5)
    ax.set_xticks(ticks=range(-20, 21, 5)) # Adjusted ticks for smaller subplot size
    ax.grid(axis='y', linestyle='--', alpha=0.7)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')
fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()
location_ids_by_coverage.remove('VLZX7K2M9QD4T')
before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                   .set_index('loc_id')
                                   ['customer_ids']
                                   .loc[location_ids_by_coverage])
location_ids_by_coverage.insert(0, 'VLZX7K2M9QD4T')

ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
axes = axes.flatten()

# --- MODIFICATION: Create a colormap for the 1-10 purchase range ---
min_purchases = 1
max_purchases = 8
num_colors = max_purchases - min_purchases + 1
colors = plt.cm.viridis(np.linspace(0, 1, num_colors))

for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off')
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off')
        continue
    
    # Prepare data for stacked bar chart
    deviation_df = pd.DataFrame({'orders': customer_orders_day})
    deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
    deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
    # Group by both deviation and the original number of orders to get the counts for each segment.
    stacked_data = deviation_df.groupby(['deviation', 'orders']).size().unstack(fill_value=0)

    # Create the stacked bar plot
    bottom = np.zeros(len(stacked_data))

    # Loop through each original order count to create the stacks.
    for order_count, counts_per_deviation in stacked_data.items():
        # --- MODIFICATION: Only plot bars for purchase counts between 1 and 10 ---
        if min_purchases <= order_count <= max_purchases:
            # Map the order count (1-10) to a color index (0-9)
            color_index = order_count - min_purchases
            color = colors[color_index]
            ax.bar(stacked_data.index, counts_per_deviation, bottom=bottom, label=f'{order_count} orders', color=color, edgecolor='white', linewidth=0.7)
            bottom += counts_per_deviation.values

    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-15.5, 15.5)
    ax.set_xticks(ticks=range(-15, 16, 5))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

# Add a single, shared legend for the entire figure
handles, labels = ax.get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, title='Original Purchases', loc='center right', bbox_to_anchor=(1.05, 0.5))

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
plt.show()
location_ids_by_coverage.remove('VLZX7K2M9QD4T')

before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                   .set_index('loc_id')
                                   ['customer_ids']
                                   .loc[location_ids_by_coverage])

location_ids_by_coverage.insert(0, 'VLZX7K2M9QD4T')

ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
axes = axes.flatten()

# --- MODIFICATION: Define colors for gender ---
gender_colors = {'male': 'cornflowerblue', 'female': 'lightcoral'}

for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off')
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off')
        continue
    
    # --- MODIFICATION: Prepare data for gender-stacked bar chart ---
    # Convert series to DataFrame and reset index to get customer_id as a column.
    deviation_df = customer_orders_day.to_frame(name='orders').reset_index()
    
    # Merge with the customers DataFrame to get gender information.
    # Assuming 'customers' DataFrame has 'customer_id' and 'gender' columns.
    deviation_df = pd.merge(deviation_df, customers, on='customer_id', how='left')

    # Calculate mean and deviation after merging.
    deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
    deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
    # Group by deviation and gender to get counts for stacking.
    stacked_data = deviation_df.groupby(['deviation', 'gender']).size().unstack(fill_value=0)
    
    # Ensure both Male and female columns exist to avoid errors.
    if 'male' not in stacked_data: stacked_data['male'] = 0
    if 'female' not in stacked_data: stacked_data['female'] = 0

    # --- MODIFICATION: Create the stacked bar plot ---
    # Plot Male bars first (the bottom layer).
    ax.bar(stacked_data.index, stacked_data['male'], color=gender_colors['male'], label='male', edgecolor='white')
    # Plot female bars on top of the Male bars.
    ax.bar(stacked_data.index, stacked_data['female'], bottom=stacked_data['male'], color=gender_colors['female'], label='female', edgecolor='white')

    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-20.5, 20.5)
    ax.set_xticks(ticks=range(-20, 21, 5))
    ax.grid(axis='y', linestyle='--', alpha=0.7)

# --- Add a single, shared legend for the entire figure ---
handles = [plt.Rectangle((0,0),1,1, color=gender_colors[label]) for label in ['male', 'female']]
labels = ['male', 'female']
fig.legend(handles, labels, title='Gender', loc='center right', bbox_to_anchor=(1.05, 0.5))

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Customer Orders Day Deviation by Gender', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
plt.show()